# TrainingWeightedTokens

Weighted experiment: decision tokens have weight 1.0; JSON structure and the closing chat separator have weight 0.1; fixed field names have weight 0.0 via labels=-100. Keep the complete assistant JSON in the input sequence. Reuse our existing InternVL model loader, image tiling, LoRA attachment and adapter saver. The new loss retains the existing causal shift and applies positional weights to unreduced cross-entropy. The joint sample reader, span builder, collator and weighted loss are in weighted_tokens_helpers.py. Existing helpers are unchanged.

Use AppliedTaskByte (.venvAppliedTask). Restart the kernel before loading the model. This notebook has not been run. The current single page is an overfit smoke experiment, not a validation result.


In [1]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "models.json").is_file():
    raise FileNotFoundError("Start in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
import importlib
import weighted_tokens_helpers
importlib.reload(weighted_tokens_helpers)
from weighted_tokens_helpers import load_joint_sample, build_masked_target, collate_weighted_tokens, weighted_supervised_loss

SAMPLE_PATHS = [ROOT / "data/training/joint_targets/seq_71033fec4de494fc_page_01/sample.json"]
samples = [load_joint_sample(path) for path in SAMPLE_PATHS]
print("Interpreter:", sys.executable)
print("Samples:", [sample["page_id"] for sample in samples])


Interpreter: c:\Users\ESHAAN\BYTE\Applied-Task\.venvAppliedTask\Scripts\python.exe
Samples: ['seq_71033fec4de494fc_page_01']


## Loss and LoRA configuration

Train every subtoken belonging to P/A/B IDs and UNKNOWN at weight 1.0. Give JSON punctuation (including empty lists and closing braces) and the closing chat separator weight 0.1. Mask fixed field names and the prompt/image/header. No padding is needed with batch size one; a future padded collator must also mask padding.

Freeze the vision encoder, projector and base LLM. Train q_proj, k_proj, v_proj and o_proj LoRA in the final eight LLM layers initially; set last_layers=None for all LLM layers. Rank 4, alpha 8 and dropout 0.05 are small starting settings for this 8 GiB GPU. Use the configurable loss weights below. Set formatting_weight and end_weight to zero to compare against the original decision-only experiment. The weighted loss is sum(weight * token_CE) / sum(weight).


In [2]:
LOSS_CONFIG = {
    "decision_weight": 1.0,
    "formatting_weight": 0.1,
    "end_weight": 0.1,
    "ignore_index": -100,
    "train_decisions": ["P IDs", "A IDs", "B IDs", "UNKNOWN"],
}
CONFIG = H.experiment(
    name="weighted_tokens_structure_smoke_01",
    decoder="attention",  # q_proj, k_proj, v_proj, o_proj
    last_layers=8,
    projector=False,
    vision=False,
    rank=4,
    epochs=100,
    lr=1e-4,
    max_tiles=1,
    accumulation=1,
)
CONFIG["view"] = "annotated"
CONFIG["max_length"] = 4096
CONFIG["loss"] = LOSS_CONFIG
RUN = ROOT / "data/training/runs" / CONFIG["name"]
print(json.dumps(CONFIG, indent=2))


{
  "name": "weighted_tokens_structure_smoke_01",
  "decoder": "attention",
  "last_layers": 8,
  "projector": false,
  "vision": false,
  "vision_last_layers": 4,
  "rank": 4,
  "alpha": 8,
  "dropout": 0.05,
  "epochs": 100,
  "lr": 0.0001,
  "max_tiles": 1,
  "view": "annotated",
  "max_length": 4096,
  "accumulation": 1,
  "seed": 42,
  "loss": {
    "decision_weight": 1.0,
    "formatting_weight": 0.1,
    "end_weight": 0.1,
    "ignore_index": -100,
    "train_decisions": [
      "P IDs",
      "A IDs",
      "B IDs",
      "UNKNOWN"
    ]
  }
}


## Inspect the mask before loading model weights

Load only local tokenizer files here. Offset mappings require the fast tokenizer. The collator also checks that its assistant token IDs exactly match the model loader's tokenizer.

Tokens overlapping a decision span receive weight 1.0 as whole tokenizer tokens. Tokens overlapping a fixed field name receive weight 0.0; all other JSON tokens receive weight 0.1. If a token mixes punctuation with an ID or field name, these components cannot be weighted separately. Every piece of a split ID receives decision weight.


In [3]:
from transformers import AutoTokenizer

offset_tokenizer = AutoTokenizer.from_pretrained(
    str(H.MODEL_DIR), trust_remote_code=True,
    local_files_only=True, use_fast=True,
)
preview = build_masked_target(offset_tokenizer, samples[0]["target"], separator="<|im_end|>", loss_config=LOSS_CONFIG)
print("Assistant JSON:", preview["text"])
print("Decision spans:", [preview["text"][start:end] for start, end in preview["spans"]])
print("\nToken                     Weight")
for weight, (start, end) in zip(preview["token_weights"], preview["offsets"]):
    piece = (preview["text"] + "<|im_end|>")[start:end]
    print(f"{piece!r:25} {weight:.2f}")


Assistant JSON: {"characters":{"P3":["A4","A5","A7"],"P2":["A6","A2"],"P1":["A1","A3"]},"rejected_appearances":[],"bubble_speakers":{"B10":"P3","B7":"P3","B8":"P3","B6":"P2","B2":"P1","B13":"P1","B9":"P1","B1":"P1","B5":"P1","B12":"P1","B4":"P1","B14":"P1","B3":"P3","B11":"P1"}}
Decision spans: ['P3', 'A4', 'A5', 'A7', 'P2', 'A6', 'A2', 'P1', 'A1', 'A3', 'B10', 'P3', 'B7', 'P3', 'B8', 'P3', 'B6', 'P2', 'B2', 'P1', 'B13', 'P1', 'B9', 'P1', 'B1', 'P1', 'B5', 'P1', 'B12', 'P1', 'B4', 'P1', 'B14', 'P1', 'B3', 'P3', 'B11', 'P1']

Token                     Weight
'{"'                      0.10
'characters'              0.00
'":{"'                    0.10
'P'                       1.00
'3'                       1.00
'":["'                    0.10
'A'                       1.00
'4'                       1.00
'","'                     0.10
'A'                       1.00
'5'                       1.00
'","'                     0.10
'A'                       1.00
'7'                       1.00
'"

## Load the local model and attach LoRA

Run this cell only when ready to use the GPU. The base model remains C:/MODELS/InternVL3-2B.


In [4]:
import random
import torch

if RUN.exists():
    raise FileExistsError(f"Choose a new run name; {RUN} already exists.")
random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])
H.cleanup()
model, tokenizer = H.load_base()
parts = H.attach_lora(model, CONFIG)
print(parts)
assert not any(p.requires_grad for p in model.vision_model.parameters())
assert not any(p.requires_grad for p in model.mlp1.parameters())

def collator(items):
    return collate_weighted_tokens(items, model, tokenizer, offset_tokenizer, CONFIG)

batch = collator([samples[0]])
assert (batch["labels"][:, :batch["answer_start"]] == -100).all()
print("Total tokens:", batch["input_ids"].numel())
assert (batch["token_weights"][:, :batch["answer_start"]] == 0).all()
print("Decision tokens:", int((batch["token_weights"] == LOSS_CONFIG["decision_weight"]).sum()))
print("All supervised tokens:", int((batch["token_weights"] > 0).sum()))
del batch


Loading local base model directly on CUDA...
FlashAttention2 is not installed.


W1004 14:51:10.279000 35320 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


[{'part': 'language_model', 'trainable': 311296}, {'part': 'mlp1', 'trainable': 0}, {'part': 'vision_model', 'trainable': 0}]
Total tokens: 923
Decision tokens: 81
All supervised tokens: 127


## Train the one-page smoke experiment

Use weighted_supervised_loss: unreduced causal cross-entropy multiplied by the collator's per-position weights and normalized by their sum. Labels at position t are predicted from hidden state t-1; weights stay aligned with those labels. The model sees the full JSON; ignored tokens still provide teacher-forced context.

The DataLoader uses our new collator and batch size one. Accumulation supports a larger effective batch. Only LoRA parameters enter the optimizer. This loop does not claim unseen-page accuracy.


In [5]:
from torch.utils.data import DataLoader

if LOSS_CONFIG["ignore_index"] != -100:
    raise ValueError("Our label mask uses ignore_index=-100.")
if RUN.exists():
    raise FileExistsError("Choose a new run name before training.")
RUN.mkdir(parents=True)
H.write_json(RUN / "config.json", CONFIG)
H.write_json(RUN / "sample_paths.json", [str(path) for path in SAMPLE_PATHS])
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=CONFIG["lr"])
loader = DataLoader(samples, batch_size=1, shuffle=True, collate_fn=collator, num_workers=0)
history = []
first_update = True
accumulation = CONFIG["accumulation"]
H.training_mode(model)

for epoch in range(CONFIG["epochs"]):
    optimizer.zero_grad(set_to_none=True)
    epoch_loss = 0.0
    for step, batch in enumerate(loader):
        loss = weighted_supervised_loss(model, batch)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss. Stop and inspect the sample/model.")
        epoch_loss += float(loss.detach())
        # Normalize a final incomplete accumulation group correctly.
        group_start = (step // accumulation) * accumulation
        group_size = min(accumulation, len(loader) - group_start)
        (loss / group_size).backward()
        if (step + 1) % accumulation == 0 or step + 1 == len(loader):
            if first_update:
                report = H.grad_report(model)
                print("First-update gradients:", report)
                if report.get("language_model", 0.0) <= 0:
                    raise RuntimeError("No language LoRA gradient.")
                first_update = False
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        del batch, loss
    row = {"epoch": epoch + 1, "weighted_token_loss": epoch_loss / len(loader)}
    history.append(row)
    print(row)
    H.write_json(RUN / "loss_history.json", history)

H.save_adapters(model, RUN, CONFIG)
print("Saved adapters:", RUN)


First-update gradients: {'language_model': 1.6816003620624542}
{'epoch': 1, 'weighted_token_loss': 0.9286318421363831}
{'epoch': 2, 'weighted_token_loss': 0.9240663051605225}
{'epoch': 3, 'weighted_token_loss': 0.9247364401817322}
{'epoch': 4, 'weighted_token_loss': 0.9180992245674133}
{'epoch': 5, 'weighted_token_loss': 0.9131392240524292}
{'epoch': 6, 'weighted_token_loss': 0.9003615975379944}
{'epoch': 7, 'weighted_token_loss': 0.8983680605888367}
{'epoch': 8, 'weighted_token_loss': 0.8953459858894348}
{'epoch': 9, 'weighted_token_loss': 0.8762830495834351}
{'epoch': 10, 'weighted_token_loss': 0.8644207715988159}
{'epoch': 11, 'weighted_token_loss': 0.8597584366798401}
{'epoch': 12, 'weighted_token_loss': 0.8470471501350403}
{'epoch': 13, 'weighted_token_loss': 0.8334119915962219}
{'epoch': 14, 'weighted_token_loss': 0.8309258222579956}
{'epoch': 15, 'weighted_token_loss': 0.810945451259613}
{'epoch': 16, 'weighted_token_loss': 0.8054434061050415}
{'epoch': 17, 'weighted_token_loss'

c:\Users\ESHAAN\BYTE\Applied-Task\.venvAppliedTask\Lib\site-packages\peft\utils\save_and_load.py:300: UserWarning: Could not find a config file in ./pretrained/Qwen2.5-32B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


## Generate and inspect the result

Check free generation and JSON termination, not just teacher-forced loss. Structural and end-token weights provide a small syntax/termination signal; they do not guarantee valid JSON. This is the same training page; held-out pages and sequence-separated splits are needed for generalization testing. Compare structural weight 0.0, 0.05 and 0.1 under the same model/data settings before selecting one.

P IDs are arbitrary, so compare identity membership and bubble relations even if generated P names differ. Exact target equality below is only an overfit diagnostic. It is not a permutation-invariant evaluation score.


In [6]:
model.eval()
sample = samples[0]
pixels = H.tiles(sample, CONFIG["max_tiles"], view="annotated")
pixels = pixels.to(device="cuda", dtype=next(model.vision_model.parameters()).dtype)
with torch.inference_mode():
    response = model.chat(tokenizer, pixels, sample["prompt"],
                          {"max_new_tokens": 1024, "do_sample": False})
print(response)
try:
    prediction = json.loads(response)
    print("Valid JSON:", True)
    print("Exact smoke target:", prediction == sample["target"])
except json.JSONDecodeError:
    prediction = None
    print("Valid JSON:", False)
H.write_json(RUN / "smoke_prediction.json", {"response": response, "prediction": prediction})
del pixels


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


{"characters":{"P3":["A4","A5","A7"],"P2":["A6","A2"],"P1":["A1","A3","A Weiner"],"P0":["A0","A8","A7"],"Piana3":["A7"],"P03":["A2"],"P33":["A3"],"P23":["A1"],"P13":["A4"],"P033":["A0"],"P333":["A7"],"P133":["A3"],"P0333":["A1"],"P233":["A2"],"P1333":["A4"],"P3333":["A7"],"P13333":["A3"],"P03333":["A1"],"P033333":["A4"],"P33333":["A7"],"P133333":["A3"],"P0333333":["A1"],"P23333":["A2"],"P1333333":["A4"],"P333333":["A7"],"P13333333":["A3"],"P03333333":["A1"],"P33333333":["A4"],"P133333333":["A3"],"P033333333":["A1"],"P2333333":["A2"],"P1333333333":["A4"],"P33333333":["A7"],"P13333333333":["A3"],"P0333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333333

References: [tokenizer offset mappings](https://huggingface.co/docs/transformers/main_classes/tokenizer), [PyTorch cross-entropy](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.cross_entropy.html), [PEFT LoRA configuration](https://huggingface.co/docs/peft/package_reference/lora).
